# SAEs Reveal Algorithmic Representations in NARs — Experiments

End-to-end pipeline:
1. **Phase 1**: Train NAR on CLRS-30 (BFS first)
2. **Phase 2**: Collect activations & train SAE
3. **Phase 3**: Feature interpretation & concept correlations

In [ ]:
# --- Colab setup (skip if running locally) ---
import os

IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in str(os.environ.get("COLAB_RELEASE_TAG", ""))

if IN_COLAB:
    if not os.path.exists("/content/nar-mechinterp"):
        !git clone https://github.com/aniervs/nar-mechinterp.git
        %cd nar-mechinterp
        # Install salsa-clrs from git first (not on PyPI, pip can't resolve it from pyproject.toml)
        !pip install salsa-clrs@git+https://github.com/jkminder/SALSA-CLRS.git 2>&1 | tail -3
        # Install the project + remaining deps
        !pip install -e . 2>&1 | tail -5
        # Force-reinstall numpy/scipy to avoid Colab's stale pre-installed versions
        !pip install --force-reinstall numpy scipy 2>&1 | tail -3
        print("Colab setup complete! Restarting runtime — re-run this cell after restart.")
        import IPython
        IPython.get_ipython().kernel.do_shutdown(restart=True)
    else:
        %cd /content/nar-mechinterp
        print("Colab setup already done.")

    # Mount Google Drive for persistent storage
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("Running locally, skipping Colab setup.")

In [ ]:
import sys
from pathlib import Path

# Handle import paths for both Colab (cwd=repo root) and local (cwd=experiments/)
if Path("../data").exists() and ".." not in sys.path:
    sys.path.insert(0, "..")
elif Path("data").exists() and "." not in sys.path:
    sys.path.insert(0, ".")

# Fix PyG pickling issue on some versions (Colab)
try:
    import torch_geometric.data.data as _pyg_data
    if not hasattr(_pyg_data, 'DataEdgeAttr'):
        _pyg_data.DataEdgeAttr = type('DataEdgeAttr', (), {})
        _pyg_data.DataTensorAttr = type('DataTensorAttr', (), {})
except Exception:
    pass

import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt
import numpy as np

from data import (
    get_clrs_dataset,
    get_clrs_dataloader,
    get_algorithm_spec,
    spec_to_model_types,
    batch_to_model_inputs,
)
from models import NARModel
from interp import (
    SparseAutoencoder, BatchTopKSAE, Transcoder,
    SAETrainer, SAEOutput,
    ActivationCollector, make_activation_dataloader,
    FeatureAnalyzer, FeatureAnalysisResult,
)
from interp.concept_labels import collect_concept_labels

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

SEED = 42
torch.manual_seed(SEED)

## Configuration

In [ ]:
# --- Experiment configuration ---
ALGORITHM = "bfs"  # Start with BFS (simplest, clearest ground-truth concepts)

# === Scale toggle ===
# Set to True for quick local testing (M1 MacBook), False for full runs on GPU
LOCAL_DEBUG = False

if LOCAL_DEBUG:
    # Small model for quick sanity checks (~30K params)
    HIDDEN_DIM = 32
    NUM_LAYERS = 2
    NUM_HEADS = 4
    NAR_EPOCHS = 20
    TRAIN_SAMPLES = 200
    VAL_SAMPLES = 64
    SAE_STEPS = 5_000
    SAE_NUM_SAMPLES = 500
else:
    # Full-scale model for real experiments (~1.5M params)
    HIDDEN_DIM = 128
    NUM_LAYERS = 4
    NUM_HEADS = 8
    NAR_EPOCHS = 100
    TRAIN_SAMPLES = 1000
    VAL_SAMPLES = 128
    SAE_STEPS = 50_000
    SAE_NUM_SAMPLES = 10_000

PROCESSOR_TYPE = "mpnn"

# Training (shared)
NAR_BATCH_SIZE = 32
NAR_LR = 1e-3
NUM_NODES = 16
EDGE_PROB = 0.2

# SAE hyperparameters
SAE_TYPE = "batchtopk"
EXPANSION_FACTOR = 4     # dict_size = HIDDEN_DIM * EXPANSION_FACTOR (reduced from 8 to cut dead features)
SAE_K = 16 if not LOCAL_DEBUG else 8  # fewer active features → sparser, more monosemantic
SAE_LR = 3e-4
SAE_BATCH_SIZE = 256

# Paths — use Google Drive on Colab for persistence
REPO_ROOT = Path("..") if Path("../data").exists() else Path(".")

if IN_COLAB:
    DRIVE_ROOT = Path("/content/drive/MyDrive/nar-mechinterp")
    CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints" / ALGORITHM
    RESULTS_DIR = DRIVE_ROOT / "results" / "sae" / ALGORITHM / SAE_TYPE
else:
    CHECKPOINT_DIR = REPO_ROOT / "checkpoints" / ALGORITHM
    RESULTS_DIR = REPO_ROOT / "results" / "sae" / ALGORITHM / SAE_TYPE

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Mode: {'LOCAL DEBUG' if LOCAL_DEBUG else 'FULL SCALE'}")
print(f"Algorithm: {ALGORITHM}")
print(f"Saving to: {CHECKPOINT_DIR}")
print(f"NAR: hidden_dim={HIDDEN_DIM}, layers={NUM_LAYERS}, heads={NUM_HEADS}, processor={PROCESSOR_TYPE}")
print(f"SAE: type={SAE_TYPE}, expansion={EXPANSION_FACTOR}x, dict_size={HIDDEN_DIM * EXPANSION_FACTOR}, k={SAE_K}")
print(f"Training: {NAR_EPOCHS} epochs, {TRAIN_SAMPLES} samples")

---
## Phase 1: Train NAR on CLRS-30

Train an MPNN-based NAR model on BFS. The model learns to predict BFS outputs (reachability) with hint supervision at each message-passing step.

### 1.1 Load CLRS-30 dataset

In [ ]:
# Load dataset and extract algorithm spec
ds = get_clrs_dataset(
    ALGORITHM, split="train",
    num_samples=TRAIN_SAMPLES,
    num_nodes=NUM_NODES,
    edge_probability=EDGE_PROB,
)
spec = get_algorithm_spec(ALGORITHM, dataset=ds)
output_types, hint_types = spec_to_model_types(spec)

print(f"Output types: {output_types}")
print(f"Hint types: {hint_types}")

# Create dataloaders
train_loader = get_clrs_dataloader(
    ALGORITHM, "train",
    batch_size=NAR_BATCH_SIZE,
    num_samples=TRAIN_SAMPLES,
    num_nodes=NUM_NODES,
    edge_probability=EDGE_PROB,
    seed=SEED,
)
val_loader = get_clrs_dataloader(
    ALGORITHM, "val",
    batch_size=NAR_BATCH_SIZE,
    num_samples=VAL_SAMPLES,
    num_nodes=NUM_NODES,
    edge_probability=EDGE_PROB,
    seed=SEED + 1,
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

### 1.2 Train NAR model

In [ ]:
model = NARModel(
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    processor_type=PROCESSOR_TYPE,
).to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"NAR model parameters: {num_params:,}")

In [ ]:
def train_epoch(model, loader, optimizer, spec, output_types, hint_types, device):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        inputs, outputs, hints = batch_to_model_inputs(batch, spec, device)
        optimizer.zero_grad()
        output = model(
            inputs=inputs, hints=hints, outputs=outputs,
            output_types=output_types, hint_types=hint_types,
            num_steps=batch.lengths.max().item(),
        )
        output.total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += output.total_loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, spec, output_types, hint_types, device):
    model.eval()
    total_loss = 0
    total_acc = 0
    count = 0
    for batch in loader:
        batch = batch.to(device)
        inputs, outputs, hints = batch_to_model_inputs(batch, spec, device)
        output = model(
            inputs=inputs, outputs=outputs,
            output_types=output_types, hint_types=hint_types,
            num_steps=batch.lengths.max().item(),
        )
        total_loss += output.total_loss.item()
        for name, pred in output.predictions.items():
            if name in outputs:
                target = outputs[name]
                if output_types.get(name) == 'node_mask':
                    pred_bin = torch.sigmoid(pred[:, :target.shape[-1]]) > 0.5
                    acc = (pred_bin == target.bool()).float().mean()
                elif output_types.get(name) == 'node_pointer':
                    pred_idx = pred.argmax(-1)
                    tgt_idx = target.long() if target.dim() < pred.dim() else target.argmax(-1)
                    acc = (pred_idx == tgt_idx).float().mean()
                else:
                    acc = torch.tensor(0.0)
                total_acc += acc.item()
                count += 1
    return total_loss / max(len(loader), 1), total_acc / max(count, 1)

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=NAR_LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=NAR_EPOCHS)

train_losses, val_losses, val_accs = [], [], []
best_loss = float('inf')

for epoch in range(NAR_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, spec, output_types, hint_types, device)
    val_loss, val_acc = validate(model, val_loader, spec, output_types, hint_types, device)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save({
            'model_state_dict': model.state_dict(),
            'epoch': epoch,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'config': {
                'hidden_dim': HIDDEN_DIM, 'num_layers': NUM_LAYERS,
                'num_heads': NUM_HEADS, 'processor_type': PROCESSOR_TYPE,
            },
        }, CHECKPOINT_DIR / "best.pt")

    print(f"Epoch {epoch+1}/{NAR_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | Acc: {val_acc:.4f}")

torch.save({'model_state_dict': model.state_dict()}, CHECKPOINT_DIR / "final.pt")

# Save training history
torch.save({
    'train_losses': train_losses,
    'val_losses': val_losses,
    'val_accs': val_accs,
}, CHECKPOINT_DIR / "training_history.pt")

print(f"\nBest val loss: {best_loss:.4f}")
print(f"Saved training history to {CHECKPOINT_DIR / 'training_history.pt'}")

### 1.3 Training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, label="Train")
ax1.plot(val_losses, label="Val")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title(f"NAR Training — {ALGORITHM.upper()}")
ax1.legend()
ax1.set_yscale("log")

ax2.plot(val_accs)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Validation Accuracy")
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

---
## Phase 2: Collect Activations & Train SAE

Collect processor hidden states from the trained NAR, then train a BatchTopK SAE to discover monosemantic features. Each (node, message-passing step) pair is treated as an independent sample.

### 2.1 Load best NAR checkpoint

In [ ]:
# Load best checkpoint
ckpt = torch.load(CHECKPOINT_DIR / "best.pt", map_location=device, weights_only=True)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded best checkpoint (epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f}, val_acc={ckpt['val_acc']:.4f})")

### 2.2 Collect processor activations

In [ ]:
# Create a larger dataset for activation collection
# IMPORTANT: shuffle=False so activation order matches concept label order
act_loader = get_clrs_dataloader(
    ALGORITHM, "train",
    batch_size=32,
    num_samples=SAE_NUM_SAMPLES,
    num_nodes=NUM_NODES,
    edge_probability=EDGE_PROB,
    seed=SEED + 100,
    shuffle=False,
)

collector = ActivationCollector(model, spec=spec, device=device)
activations = collector.collect(act_loader, output_types=output_types)
print(f"Collected activations: {activations.shape}")
print(f"  = {activations.shape[0]:,} activation vectors of dim {activations.shape[1]}")

# Save activations
torch.save(activations.cpu(), RESULTS_DIR / "activations.pt")
print(f"Saved to {RESULTS_DIR / 'activations.pt'}")

In [ ]:
activations = torch.load(RESULTS_DIR / "activations.pt")

### 2.3 Collect concept labels

Extract ground-truth algorithmic concept labels (is_visited, is_frontier, is_source, etc.) aligned with the activation vectors.

In [ ]:
# Recreate dataloader with same seed (no shuffle) so labels align with activations
label_loader = get_clrs_dataloader(
    ALGORITHM, "train",
    batch_size=32,
    num_samples=SAE_NUM_SAMPLES,
    num_nodes=NUM_NODES,
    edge_probability=EDGE_PROB,
    seed=SEED + 100,
    shuffle=False,
)

concept_result = collect_concept_labels(label_loader, spec, ALGORITHM)
print(f"Concept labels collected: {concept_result.num_samples} samples")
print(f"Concepts: {list(concept_result.labels.keys())}")
for name, desc in concept_result.concept_descriptions.items():
    print(f"  {name}: {desc}")

# Save concept labels
torch.save({
    "labels": concept_result.labels,
    "descriptions": concept_result.concept_descriptions,
    "algorithm": concept_result.algorithm,
    "num_samples": concept_result.num_samples,
}, RESULTS_DIR / "concept_labels.pt")
print(f"\nSaved to {RESULTS_DIR / 'concept_labels.pt'}")

### 2.4 Train BatchTopK SAE

In [ ]:
dict_size = HIDDEN_DIM * EXPANSION_FACTOR

sae = BatchTopKSAE(
    input_dim=HIDDEN_DIM,
    dict_size=dict_size,
    k=SAE_K,
).to(device)

print(f"BatchTopK SAE: input_dim={HIDDEN_DIM}, dict_size={dict_size}, k={SAE_K}")
print(f"SAE parameters: {sum(p.numel() for p in sae.parameters()):,}")

In [ ]:
trainer = SAETrainer(sae, lr=SAE_LR, resample_dead_every=0)
sae_loader = make_activation_dataloader(activations, batch_size=SAE_BATCH_SIZE)

sae_losses = []
step = 0

while step < SAE_STEPS:
    for (batch,) in sae_loader:
        if step >= SAE_STEPS:
            break
        batch = batch.to(device)
        output = trainer.train_step(batch)

        if step % 500 == 0:
            sae_losses.append({
                'step': step,
                'loss': output.loss.item(),
                'recon': output.reconstruction_loss.item(),
                'l0': output.l0.item(),
            })
            print(f"Step {step}/{SAE_STEPS} | loss={output.loss.item():.4f} | recon={output.reconstruction_loss.item():.4f} | L0={output.l0.item():.1f}")

        # Save checkpoint every 10K steps
        if step > 0 and step % 10_000 == 0:
            torch.save({
                "state_dict": sae.state_dict(),
                "config": sae.get_config(),
                "sae_type": SAE_TYPE,
                "step": step,
            }, RESULTS_DIR / f"sae_step{step}.pt")
            print(f"  [checkpoint saved: step {step}]")

        step += 1

print(f"\nFinal training stats: {trainer.get_training_stats()}")

# Save SAE training losses
torch.save(sae_losses, RESULTS_DIR / "sae_losses.pt")
print(f"Saved training losses to {RESULTS_DIR / 'sae_losses.pt'}")

In [ ]:
# Save SAE checkpoint
torch.save({
    "state_dict": sae.state_dict(),
    "config": sae.get_config(),
    "sae_type": SAE_TYPE,
}, RESULTS_DIR / "sae.pt")
print(f"Saved SAE to {RESULTS_DIR / 'sae.pt'}")

In [ ]:
ckpt = torch.load(RESULTS_DIR / "sae.pt", map_location=device, weights_only=True)
sae.load_state_dict(ckpt["state_dict"])
sae.eval()
print(f"Loaded SAE from {RESULTS_DIR / 'sae.pt'}")

### 2.5 SAE training curves

In [ ]:
steps = [d['step'] for d in sae_losses]
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))

ax1.plot(steps, [d['loss'] for d in sae_losses])
ax1.set_xlabel("Step")
ax1.set_ylabel("Total Loss")
ax1.set_title("SAE Total Loss")

ax2.plot(steps, [d['recon'] for d in sae_losses])
ax2.set_xlabel("Step")
ax2.set_ylabel("Reconstruction Loss")
ax2.set_title("SAE Reconstruction Loss")

ax3.plot(steps, [d['l0'] for d in sae_losses])
ax3.set_xlabel("Step")
ax3.set_ylabel("L0 (active features)")
ax3.set_title("SAE Sparsity (L0)")
ax3.axhline(y=SAE_K, color='r', linestyle='--', alpha=0.5, label=f"target k={SAE_K}")
ax3.legend()

plt.tight_layout()
plt.show()

---
## Phase 3: Feature Interpretation

Analyze SAE features: basic statistics, reconstruction quality, and — most importantly — correlations with ground-truth algorithmic concepts (is_visited, is_frontier, etc.).

### 3.1 Reconstruction quality & dead features

In [ ]:
sae.eval()
analyzer = FeatureAnalyzer(sae)

with torch.no_grad():
    sample = activations[:10_000].to(device)
    # Evaluate in same batch size as training to match BatchTopK behavior
    recon_list = []
    for i in range(0, len(sample), SAE_BATCH_SIZE):
        batch = sample[i:i+SAE_BATCH_SIZE]
        out = sae(batch)
        recon_list.append(out.reconstructed)
    reconstructed = torch.cat(recon_list)

    recon_mse = F.mse_loss(reconstructed, sample).item()
    total_var = sample.var(dim=0).sum()
    residual_var = (sample - reconstructed).var(dim=0).sum()
    fve = (1 - residual_var / total_var).item()

print(f"Reconstruction MSE: {recon_mse:.6f}")
print(f"Fraction of variance explained: {fve:.4f}")

# Dead features (also batched)
with torch.no_grad():
    all_features = []
    for i in range(0, len(sample), SAE_BATCH_SIZE):
        batch = sample[i:i+SAE_BATCH_SIZE]
        out = sae(batch)
        all_features.append(out.features)
    features = torch.cat(all_features)
    num_dead = (features.sum(dim=0) == 0).sum().item()
print(f"Dead features: {num_dead}/{dict_size} ({100*num_dead/dict_size:.1f}%)")

l0_per_sample = (features > 0).float().sum(dim=-1)
print(f"L0 (avg active features): {l0_per_sample.mean():.1f} +/- {l0_per_sample.std():.1f}")
print(f"L0 range: [{l0_per_sample.min():.0f}, {l0_per_sample.max():.0f}]")


### 3.2 Feature statistics

In [ ]:
stats = analyzer.compute_feature_stats(activations.to(device))
stats_sorted = sorted(stats, key=lambda s: s.activation_frequency, reverse=True)

# Feature frequency distribution
freqs = [s.activation_frequency for s in stats]
active_freqs = [f for f in freqs if f > 0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(freqs, bins=50, edgecolor='black', alpha=0.7)
ax1.set_xlabel("Activation Frequency")
ax1.set_ylabel("Count")
ax1.set_title("Feature Activation Frequency Distribution")
ax1.axvline(x=0, color='r', linestyle='--', alpha=0.5)

if active_freqs:
    ax2.hist(active_freqs, bins=50, edgecolor='black', alpha=0.7)
    ax2.set_xlabel("Activation Frequency")
    ax2.set_ylabel("Count")
    ax2.set_title("Active Features Only")

plt.tight_layout()
plt.show()

# Top-20 most active features
print("Top-20 most active features:")
for s in stats_sorted[:20]:
    print(f"  Feature {s.feature_idx:4d}: freq={s.activation_frequency:.4f}, "
          f"mean_act={s.mean_activation:.4f}, max_act={s.max_activation:.4f}")

### 3.3 Concept correlations — the core result

For each SAE feature, compute Pearson correlation with ground-truth BFS concepts:
- **is_visited**: node has been reached
- **is_frontier**: node just entered the wavefront this step
- **is_source**: source node
- **is_unvisited**: node not yet reached

In [ ]:
# Load concept labels
concept_data = torch.load(RESULTS_DIR / "concept_labels.pt", map_location=device, weights_only=True)
concept_labels = concept_data["labels"]

# Drop redundant concepts (is_unvisited = 1 - is_visited)
concept_labels.pop("is_unvisited", None)

# Align dimensions
N_CORR = 10_000
min_n = min(N_CORR, activations.shape[0], min(v.shape[0] for v in concept_labels.values()))

concept_names = list(concept_labels.keys())
labels = torch.stack([concept_labels[name][:min_n].float().cpu() for name in concept_names], dim=1)

print(f"Samples: {min_n:,}")
print(f"Concepts: {concept_names}")

# Batch-encode features on GPU, collect on CPU
torch.cuda.empty_cache()
all_features = []
with torch.no_grad():
    for i in range(0, min_n, SAE_BATCH_SIZE):
        batch = activations[i:i+SAE_BATCH_SIZE].to(device)
        out = sae(batch)
        all_features.append(out.features.cpu())
        del out, batch
        torch.cuda.empty_cache()
features = torch.cat(all_features).float()
features = features[:min_n]
del all_features

# Correlations on CPU
feat_centered = features - features.mean(dim=0, keepdim=True)
label_centered = labels - labels.mean(dim=0, keepdim=True)
feat_norm = feat_centered / feat_centered.std(dim=0, keepdim=True).clamp(min=1e-8)
label_norm = label_centered / label_centered.std(dim=0, keepdim=True).clamp(min=1e-8)
concept_matrix = (feat_norm.T @ label_norm) / min_n
del feat_centered, label_centered, feat_norm, label_norm

# Identify monosemantic features (relaxed: allow anti-correlation with complement)
mono = []
for i in range(dict_size):
    corrs = concept_matrix[i].abs()
    if corrs.max() > 0.5 and (corrs > 0.3).sum() <= 2:
        mono.append(i)

dead = (features.sum(dim=0) == 0).nonzero(as_tuple=True)[0].tolist()

print(f"\nMonosemantic features: {len(mono)} / {dict_size}")
print(f"Dead features: {len(dead)} / {dict_size}")

# Save correlation results
torch.save({
    "concept_matrix": concept_matrix,
    "concept_names": concept_names,
    "mono": mono,
    "dead": dead,
    "features": features,
}, RESULTS_DIR / "correlation_results.pt")
print(f"Saved correlation results to {RESULTS_DIR / 'correlation_results.pt'}")

In [ ]:
# Top correlated features per concept
for j, concept_name in enumerate(concept_names):
    corrs = concept_matrix[:, j]
    top_indices = corrs.abs().topk(5).indices
    print(f"\nConcept '{concept_name}' — top correlated features:")
    for idx in top_indices:
        print(f"  Feature {idx.item():4d}: correlation = {corrs[idx]:+.4f}")


### 3.4 Concept correlation heatmap

In [ ]:
# Heatmap of concept correlations for the top-K most interesting features
cm_np = concept_matrix.cpu().numpy()  # (dict_size, num_concepts)
max_corr_per_feat = np.abs(cm_np).max(axis=1)
top_k_feats = 30
top_feat_idxs = np.argsort(max_corr_per_feat)[-top_k_feats:][::-1]

fig, ax = plt.subplots(figsize=(8, 10))
im = ax.imshow(cm_np[top_feat_idxs], cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)

ax.set_yticks(range(len(top_feat_idxs)))
ax.set_yticklabels([f"Feature {i}" for i in top_feat_idxs])
ax.set_xticks(range(len(concept_names)))
ax.set_xticklabels(concept_names, rotation=45, ha='right')
ax.set_title(f"Top-{top_k_feats} Features x BFS Concepts — Pearson Correlation")

plt.colorbar(im, ax=ax, label="Correlation")
plt.tight_layout()
plt.show()


### 3.5 Monosemanticity analysis

A feature is **monosemantic** if it correlates strongly with exactly one concept. This is the key test: do SAE features cleanly separate algorithmic concepts?

In [ ]:
# Categorize features (using mono, dead, concept_matrix from earlier cells)
poly_feats = [i for i in range(dict_size) 
              if i not in mono and i not in dead 
              and max_corr_per_feat[i] > 0.1]
other_feats = [i for i in range(dict_size) 
               if i not in mono and i not in dead and i not in poly_feats]

print(f"Feature breakdown ({dict_size} total):")
print(f"  Monosemantic (|corr| > 0.5 with exactly 1 concept): {len(mono)}")
print(f"  Polysemantic (|corr| > 0.1 with multiple concepts):  {len(poly_feats)}")
print(f"  Weak/uncorrelated:                                    {len(other_feats)}")
print(f"  Dead (never activate):                                {len(dead)}")

# Detailed look at monosemantic features
if mono:
    print(f"\nMonosemantic features detail:")
    for feat_idx in mono[:15]:
        corrs = {name: concept_matrix[feat_idx, j].item()
                 for j, name in enumerate(concept_names)}
        best_concept = max(corrs, key=lambda k: abs(corrs[k]))
        print(f"  Feature {feat_idx:4d} -> {best_concept} (r={corrs[best_concept]:+.3f})")


### 3.6 Max correlation distribution

Distribution of the maximum absolute correlation each feature achieves with any concept. High values = features that cleanly track algorithmic state.

In [ ]:
# Filter out dead features for the distribution
active_max_corrs = max_corr_per_feat[[i for i in range(dict_size) if i not in dead]]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(active_max_corrs, bins=50, edgecolor='black', alpha=0.7)
ax.axvline(x=0.5, color='r', linestyle='--', alpha=0.7, label="Monosemantic threshold (0.5)")
ax.axvline(x=0.3, color='orange', linestyle='--', alpha=0.7, label="Weak threshold (0.3)")
ax.set_xlabel("Max |correlation| with any concept")
ax.set_ylabel("Number of features")
ax.set_title("Feature-Concept Correlation Strength (active features)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
for j, name in enumerate(concept_names):
    top_idx = concept_matrix[:, j].abs().topk(3).indices
    print(f"\n{name}:")
    for idx in top_idx:
        row = {n: concept_matrix[idx, k].item() for k, n in enumerate(concept_names)}
        print(f"  Feature {idx.item():4d}: {row}")


---

## 4. SAE Hyperparameter Sweep

Compare different SAE architectures and hyperparameters using the same NAR activations.
We sweep over:
- **Expansion factors**: 1x, 4x, 16x (dict_size = hidden_dim * factor)
- **k values**: 8, 16, 32, 64
- **SAE types**: BatchTopK vs Standard (L1-penalized)

In [ ]:
# --- Sweep configuration ---
SWEEP_STEPS = SAE_STEPS  # same training budget for fair comparison

sweep_configs = [
    # BatchTopK: vary expansion factor and k
    {"name": "btk_1x_k8",  "type": "batchtopk", "expansion": 1,  "k": 8},
    {"name": "btk_1x_k16", "type": "batchtopk", "expansion": 1,  "k": 16},
    {"name": "btk_4x_k8",  "type": "batchtopk", "expansion": 4,  "k": 8},
    {"name": "btk_4x_k16", "type": "batchtopk", "expansion": 4,  "k": 16},
    {"name": "btk_4x_k32", "type": "batchtopk", "expansion": 4,  "k": 32},
    {"name": "btk_4x_k64", "type": "batchtopk", "expansion": 4,  "k": 64},
    {"name": "btk_16x_k16","type": "batchtopk", "expansion": 16, "k": 16},
    {"name": "btk_16x_k32","type": "batchtopk", "expansion": 16, "k": 32},
    {"name": "btk_16x_k64","type": "batchtopk", "expansion": 16, "k": 64},
    # Standard SAE baselines (L1-penalized)
    {"name": "std_1x",  "type": "standard", "expansion": 1,  "sparsity_coeff": 1e-3},
    {"name": "std_4x",  "type": "standard", "expansion": 4,  "sparsity_coeff": 1e-3},
    {"name": "std_16x", "type": "standard", "expansion": 16, "sparsity_coeff": 1e-3},
]

print(f"Sweep: {len(sweep_configs)} configurations, {SWEEP_STEPS} steps each")
for cfg in sweep_configs:
    ds = HIDDEN_DIM * cfg["expansion"]
    print(f"  {cfg['name']:15s} | {cfg['type']:10s} | dict_size={ds:5d} | {'k='+str(cfg.get('k','')) if 'k' in cfg else 'L1='+str(cfg['sparsity_coeff'])}")

### 4.1 Run sweep

In [ ]:
# Load concept labels once (reuse across all configs)
concept_data = torch.load(RESULTS_DIR / "concept_labels.pt", map_location="cpu", weights_only=True)
concept_labels_sweep = concept_data["labels"]
concept_labels_sweep.pop("is_unvisited", None)
concept_names_sweep = list(concept_labels_sweep.keys())

N_CORR = 10_000
min_n = min(N_CORR, activations.shape[0], min(v.shape[0] for v in concept_labels_sweep.values()))
labels_sweep = torch.stack([concept_labels_sweep[name][:min_n].float() for name in concept_names_sweep], dim=1)

sweep_results = []
sweep_dir = RESULTS_DIR.parent.parent / "sweep"
sweep_dir.mkdir(parents=True, exist_ok=True)

for cfg in sweep_configs:
    name = cfg["name"]
    ds = HIDDEN_DIM * cfg["expansion"]
    print(f"\n{'='*60}")
    print(f"Training: {name} (dict_size={ds})")
    print(f"{'='*60}")

    # Create SAE
    if cfg["type"] == "batchtopk":
        sae_sweep = BatchTopKSAE(input_dim=HIDDEN_DIM, dict_size=ds, k=cfg["k"]).to(device)
    else:
        sae_sweep = SparseAutoencoder(input_dim=HIDDEN_DIM, dict_size=ds, sparsity_coeff=cfg["sparsity_coeff"]).to(device)

    n_params = sum(p.numel() for p in sae_sweep.parameters())
    print(f"  Parameters: {n_params:,}")

    # Train
    trainer_sweep = SAETrainer(sae_sweep, lr=SAE_LR, resample_dead_every=0)
    sae_loader_sweep = make_activation_dataloader(activations, batch_size=SAE_BATCH_SIZE)

    step = 0
    while step < SWEEP_STEPS:
        for (batch,) in sae_loader_sweep:
            if step >= SWEEP_STEPS:
                break
            batch = batch.to(device)
            output = trainer_sweep.train_step(batch)
            step += 1
    final_stats = trainer_sweep.get_training_stats()
    print(f"  Final loss: {final_stats.get('recent_loss', 0):.4f}, L0: {final_stats.get('recent_l0', 0):.1f}")

    # Evaluate reconstruction
    sae_sweep.eval()
    torch.cuda.empty_cache()
    recon_list, feat_list = [], []
    with torch.no_grad():
        for i in range(0, min_n, SAE_BATCH_SIZE):
            batch = activations[i:i+SAE_BATCH_SIZE].to(device)
            out = sae_sweep(batch)
            recon_list.append(out.reconstructed.cpu())
            feat_list.append(out.features.cpu())
            del out, batch
    reconstructed = torch.cat(recon_list)[:min_n]
    features_sweep = torch.cat(feat_list)[:min_n].float()
    del recon_list, feat_list

    sample = activations[:min_n]
    recon_mse = F.mse_loss(reconstructed, sample).item()
    total_var = sample.var(dim=0).sum()
    residual_var = (sample - reconstructed).var(dim=0).sum()
    fve = (1 - residual_var / total_var).item()
    del reconstructed

    num_dead = (features_sweep.sum(dim=0) == 0).sum().item()
    l0_per = (features_sweep > 0).float().sum(dim=-1)
    l0_mean = l0_per.mean().item()

    # Concept correlations
    feat_c = features_sweep - features_sweep.mean(dim=0, keepdim=True)
    lab_c = labels_sweep - labels_sweep.mean(dim=0, keepdim=True)
    feat_n = feat_c / feat_c.std(dim=0, keepdim=True).clamp(min=1e-8)
    lab_n = lab_c / lab_c.std(dim=0, keepdim=True).clamp(min=1e-8)
    cm = (feat_n.T @ lab_n) / min_n
    del feat_c, lab_c, feat_n, lab_n

    # Monosemantic count (relaxed criterion)
    n_mono = 0
    for i in range(ds):
        corrs = cm[i].abs()
        if corrs.max() > 0.5 and (corrs > 0.3).sum() <= 2:
            n_mono += 1

    # Max correlation per concept
    max_corrs = {concept_names_sweep[j]: cm[:, j].abs().max().item() for j in range(len(concept_names_sweep))}

    result = {
        "name": name,
        "type": cfg["type"],
        "expansion": cfg["expansion"],
        "k": cfg.get("k", None),
        "sparsity_coeff": cfg.get("sparsity_coeff", None),
        "dict_size": ds,
        "params": n_params,
        "mse": recon_mse,
        "fve": fve,
        "dead_pct": 100 * num_dead / ds,
        "l0": l0_mean,
        "n_mono": n_mono,
        "max_corrs": max_corrs,
        "final_loss": final_stats.get("recent_loss", 0),
    }
    sweep_results.append(result)
    del features_sweep, sae_sweep, cm
    torch.cuda.empty_cache()

    print(f"  FVE: {fve:.4f} | Dead: {num_dead}/{ds} ({100*num_dead/ds:.1f}%) | L0: {l0_mean:.1f} | Mono: {n_mono}")
    print(f"  Max |corr|: {max_corrs}")

    # Save checkpoint
    torch.save(result, sweep_dir / f"{name}_result.pt")

# Save all sweep results
torch.save(sweep_results, sweep_dir / "all_results.pt")
print(f"\nSaved all sweep results to {sweep_dir / 'all_results.pt'}")

### 4.2 Sweep results summary

In [ ]:
# --- Summary table ---
# Load from disk if needed: sweep_results = torch.load(sweep_dir / "all_results.pt")

print(f"{'Config':15s} | {'Type':10s} | {'DictSz':>6s} | {'FVE':>6s} | {'Dead%':>5s} | {'L0':>5s} | {'Mono':>4s} | {'MaxCorr':>7s}")
print("-" * 85)
for r in sweep_results:
    best_corr = max(r["max_corrs"].values())
    print(f"{r['name']:15s} | {r['type']:10s} | {r['dict_size']:6d} | {r['fve']:.4f} | {r['dead_pct']:5.1f} | {r['l0']:5.1f} | {r['n_mono']:4d} | {best_corr:.4f}")

### 4.3 Sweep visualizations

In [ ]:
# --- Sweep plots ---
btk = [r for r in sweep_results if r["type"] == "batchtopk"]
std = [r for r in sweep_results if r["type"] == "standard"]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. FVE vs dict_size
ax = axes[0, 0]
for exp in [1, 4, 16]:
    pts = [r for r in btk if r["expansion"] == exp]
    if pts:
        ax.scatter([r["k"] for r in pts], [r["fve"] for r in pts], label=f"BTK {exp}x", s=60)
for r in std:
    ax.scatter(r["l0"], r["fve"], marker="x", s=80, c="black", zorder=5)
ax.set_xlabel("k (or L0 for standard)")
ax.set_ylabel("FVE")
ax.set_title("Reconstruction Quality")
ax.legend()

# 2. Dead features % vs dict_size
ax = axes[0, 1]
for exp in [1, 4, 16]:
    pts = [r for r in btk if r["expansion"] == exp]
    if pts:
        ax.scatter([r["k"] for r in pts], [r["dead_pct"] for r in pts], label=f"BTK {exp}x", s=60)
for r in std:
    ax.scatter(r["l0"], r["dead_pct"], marker="x", s=80, c="black", zorder=5)
ax.set_xlabel("k (or L0 for standard)")
ax.set_ylabel("Dead features (%)")
ax.set_title("Feature Utilization")
ax.legend()

# 3. Monosemantic features vs config
ax = axes[0, 2]
names = [r["name"] for r in sweep_results]
monos = [r["n_mono"] for r in sweep_results]
colors = ["tab:blue" if r["type"] == "batchtopk" else "tab:orange" for r in sweep_results]
ax.barh(range(len(names)), monos, color=colors)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=8)
ax.set_xlabel("Monosemantic features")
ax.set_title("Monosemanticity")

# 4. Max correlation per concept
ax = axes[1, 0]
for j, concept in enumerate(concept_names_sweep):
    vals = [r["max_corrs"][concept] for r in sweep_results]
    ax.bar([i + j*0.25 for i in range(len(sweep_results))], vals, width=0.25, label=concept)
ax.set_xticks(range(len(sweep_results)))
ax.set_xticklabels([r["name"] for r in sweep_results], rotation=45, ha="right", fontsize=7)
ax.set_ylabel("Max |correlation|")
ax.set_title("Concept Alignment")
ax.legend(fontsize=8)

# 5. FVE vs monosemantic features (Pareto front)
ax = axes[1, 1]
for r in btk:
    ax.scatter(r["fve"], r["n_mono"], s=60, c="tab:blue")
    ax.annotate(r["name"], (r["fve"], r["n_mono"]), fontsize=7, ha="left")
for r in std:
    ax.scatter(r["fve"], r["n_mono"], s=60, c="tab:orange", marker="x")
    ax.annotate(r["name"], (r["fve"], r["n_mono"]), fontsize=7, ha="left")
ax.set_xlabel("FVE (reconstruction)")
ax.set_ylabel("Monosemantic features")
ax.set_title("Reconstruction vs Interpretability")

# 6. Dead % vs monosemantic
ax = axes[1, 2]
for r in btk:
    ax.scatter(r["dead_pct"], r["n_mono"], s=60, c="tab:blue")
    ax.annotate(r["name"], (r["dead_pct"], r["n_mono"]), fontsize=7, ha="left")
for r in std:
    ax.scatter(r["dead_pct"], r["n_mono"], s=60, c="tab:orange", marker="x")
    ax.annotate(r["name"], (r["dead_pct"], r["n_mono"]), fontsize=7, ha="left")
ax.set_xlabel("Dead features (%)")
ax.set_ylabel("Monosemantic features")
ax.set_title("Feature Utilization vs Interpretability")

plt.tight_layout()
plt.savefig(sweep_dir / "sweep_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved plots to {sweep_dir / 'sweep_plots.png'}")

---
## Summary & Next Steps

**Key metrics to check:**
- NAR validation accuracy > 0.9 (BFS should be near-perfect)
- SAE fraction of variance explained > 0.9
- Dead features < 30%
- Monosemantic features > 0 (ideally 10+)

**Next experiments to run:**
- Repeat for Dijkstra, DFS, MST (change `ALGORITHM` at the top)
- Multi-algorithm training (shared processor across BFS + Dijkstra + DFS)
- Per-step concept correlations (do features track temporal BFS concepts?)
- Collect activations from different processor layers separately